# Clustering & Marker Gene Identification

Cluster cells and identify marker genes per cluster.

This notebook:
1. Runs Leiden clustering at a configurable resolution
2. Identifies marker genes per cluster (Wilcoxon rank-sum test)
3. Generates UMAP, dot plots, and heatmaps
4. Saves clustered h5ad and marker tables

In [ ]:
# Parameters (injected by Papermill)
input_h5ad_path: str = "/data/results/experiment/02_normalized_dimreduced.h5ad"
experiment_name: str = "experiment"
clustering_resolution: float = 1.0
n_marker_genes: int = 25
bioaf_api_url: str = "http://localhost:8000"
experiment_id: str = ""

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False, figsize=(6, 4))

results_dir = f"/data/results/{experiment_name}"
figures_dir = os.path.join(results_dir, "figures", "clustering")
os.makedirs(figures_dir, exist_ok=True)

## 1. Load Data

In [ ]:
adata = sc.read_h5ad(input_h5ad_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Available embeddings: {list(adata.obsm.keys())}")

## 2. Leiden Clustering

In [ ]:
sc.tl.leiden(adata, resolution=clustering_resolution, key_added="leiden")
n_clusters = adata.obs["leiden"].nunique()
print(f"Found {n_clusters} clusters at resolution {clustering_resolution}")
print(adata.obs["leiden"].value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(adata, color="leiden", legend_loc="on data", ax=ax, show=False,
           title=f"Leiden Clustering (res={clustering_resolution})")
fig.savefig(os.path.join(figures_dir, "umap_clusters.png"), bbox_inches="tight")
plt.show()

## 3. Marker Gene Identification

Wilcoxon rank-sum test comparing each cluster against the rest.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon",
                        n_genes=n_marker_genes, use_raw=True)
print(f"Top {n_marker_genes} markers computed per cluster")

In [ ]:
# Summary table of top markers per cluster
marker_df = sc.get.rank_genes_groups_df(adata, group=None)
marker_df.head(20)

## 4. Marker Visualizations

In [ ]:
# Rank genes groups plot
fig = sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, show=False,
                               return_fig=True)
fig.savefig(os.path.join(figures_dir, "marker_genes_ranks.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Dot plot of top 5 markers per cluster
top_markers = marker_df.groupby("group").head(5)["names"].unique().tolist()
fig = sc.pl.dotplot(adata, var_names=top_markers[:30], groupby="leiden",
                    use_raw=True, show=False, return_fig=True)
fig.savefig(os.path.join(figures_dir, "dotplot_markers.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Heatmap of top markers
sc.pl.rank_genes_groups_heatmap(adata, n_genes=5, groupby="leiden",
                                 use_raw=True, show=False,
                                 figsize=(12, max(4, n_clusters * 0.5)))
plt.savefig(os.path.join(figures_dir, "heatmap_markers.png"), bbox_inches="tight")
plt.show()

## 5. Save Results

In [ ]:
# Save h5ad
output_path = os.path.join(results_dir, "03_clustered.h5ad")
adata.write_h5ad(output_path)

# Save marker table as CSV
markers_csv = os.path.join(results_dir, "03_marker_genes.csv")
marker_df.to_csv(markers_csv, index=False)

print(f"Saved clustered data to: {output_path}")
print(f"Saved marker genes to: {markers_csv}")
print(f"Clusters: {n_clusters}, Markers per cluster: {n_marker_genes}")